# Fiche — Stable-Baselines3 (SB3)

Stable-Baselines3 est une bibliothèque Python qui **implémente les algorithmes RL classiques** de façon fiable et simple à utiliser.

Au lieu de coder PPO ou SAC depuis zéro (ce qui est très complexe), SB3 te donne des implémentations testées et prêtes à l'emploi.

---
## Vue d'ensemble

```
TON CODE
   │
   ├── Gymnasium  →  définit l'environnement (observations, actions, rewards)
   │
   └── SB3        →  fournit les algorithmes (PPO, A2C, SAC…)
                      ↕ interagit avec Gymnasium automatiquement
```

Tu n'as pas à écrire la boucle d'entraînement, la gestion du replay buffer, ou le calcul des gradients. SB3 s'en occupe.

---
## 1. Créer un modèle

In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO

env = gym.make('LunarLander-v3')

# Les 3 arguments obligatoires : policy, env, verbose
model = PPO(
    'MlpPolicy',  # architecture du réseau de neurones
    env,          # l'environnement
    verbose=1,    # 0=silencieux, 1=affiche les métriques, 2=très verbeux
)

print("Modèle créé")
print(f"Policy : {model.policy}")

### Les 3 policies disponibles

| Policy | Observation attendue | Exemple |
|--------|---------------------|---------|
| `MlpPolicy` | Vecteur de nombres | LunarLander (8 valeurs) |
| `CnnPolicy` | Image (pixels) | Atari Breakout |
| `MultiInputPolicy` | Dict avec plusieurs types | Image + capteurs |

Dans 90% des cas, tu utiliseras `MlpPolicy`.

---
## 2. Entraîner

In [ ]:
# total_timesteps = nombre total d'interactions agent-environnement
model.learn(total_timesteps=50_000)

# Avec logs TensorBoard
# model.learn(total_timesteps=50_000, tb_log_name="mon_run")

# Pour continuer l'entraînement sans remettre le compteur à zéro
# model.learn(total_timesteps=50_000, reset_num_timesteps=False)

### Lire les métriques affichées pendant `learn()`

```
| rollout/           |          |
|    ep_len_mean     |      200 |   ← longueur moyenne des épisodes
|    ep_rew_mean     |    -134  |   ← récompense moyenne (à maximiser)
| time/              |          |
|    fps             |     1200 |   ← steps par seconde
|    total_timesteps |    10240 |   ← steps faits jusqu'ici
| train/             |          |
|    entropy_loss    |    -1.38 |   ← entropie (exploration)
|    value_loss      |     45.2 |   ← erreur du réseau de valeur (doit baisser)
```

La métrique principale à surveiller : **`ep_rew_mean`** — elle doit augmenter au fil du temps.

---
## 3. Évaluer

In [ ]:
from stable_baselines3.common.evaluation import evaluate_policy

# Lance n_eval_episodes épisodes complets et retourne mean / std
mean_reward, std_reward = evaluate_policy(
    model,
    env,
    n_eval_episodes=10,
    deterministic=True,  # l'agent choisit la meilleure action (pas aléatoire)
)

print(f"Reward moyen : {mean_reward:.2f} ± {std_reward:.2f}")

# Interpréter le résultat pour LunarLander :
# < -200 : très mauvais (crashe souvent)
# -200 à 0 : apprend à voler mais ne pose pas
# 0 à 100 : commence à atterrir
# > 200 : atterrissage réussi régulièrement

---
## 4. Prédire (utiliser le modèle)

In [ ]:
obs, info = env.reset()

for step in range(300):
    # predict() retourne toujours (action, état_interne)
    # l'état interne (_states) est utile seulement pour les RNN, sinon on l'ignore
    action, _states = model.predict(obs, deterministic=True)

    obs, reward, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        obs, info = env.reset()

env.close()
print("Épisode de test terminé")

### `deterministic=True` vs `deterministic=False`

| Mode | Comportement | Quand l'utiliser |
|------|-------------|------------------|
| `True` | Choisit toujours la meilleure action connue | Évaluation, démo finale |
| `False` | Ajoute de l'aléatoire (explore) | Pendant l'entraînement |

Pour observer l'agent → `deterministic=True`  
`evaluate_policy` utilise `True` par défaut

---
## 5. Sauvegarder et charger

In [ ]:
import os
os.makedirs("models/demo", exist_ok=True)

# Sauvegarder
model.save("models/demo/ppo_lunarlander")
# → crée le fichier models/demo/ppo_lunarlander.zip

print("Sauvegardé :", os.listdir("models/demo"))

In [ ]:
env = gym.make('LunarLander-v3')

# Charger (il faut repasser l'env pour que predict() fonctionne)
loaded_model = PPO.load("models/demo/ppo_lunarlander", env=env)

mean, std = evaluate_policy(loaded_model, env, n_eval_episodes=5)
print(f"Modèle chargé — reward moyen : {mean:.2f} ± {std:.2f}")

env.close()

---
## 6. Callbacks — automatiser des actions pendant l'entraînement

Les **callbacks** sont des fonctions que SB3 appelle automatiquement à certains moments (chaque step, chaque rollout, etc.).

Le plus utile : `EvalCallback` — évalue le modèle régulièrement et sauvegarde le meilleur.

In [ ]:
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback

env = gym.make('LunarLander-v3')
eval_env = gym.make('LunarLander-v3')  # env séparé pour l'évaluation

# EvalCallback : évalue toutes les N steps, sauvegarde le meilleur modèle
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path="models/best",
    eval_freq=5_000,         # évaluer toutes les 5000 steps
    n_eval_episodes=5,
    verbose=1,
)

# CheckpointCallback : sauvegarde régulièrement (indépendamment des perfs)
checkpoint_callback = CheckpointCallback(
    save_freq=10_000,
    save_path="models/checkpoints",
    name_prefix="ppo",
)

model = PPO('MlpPolicy', env, verbose=0)

# Passer un ou plusieurs callbacks
model.learn(
    total_timesteps=30_000,
    callback=[eval_callback, checkpoint_callback],
)

env.close()
eval_env.close()
print("Entraînement avec callbacks terminé")

### Callbacks disponibles

| Callback | Rôle |
|----------|------|
| `EvalCallback` | Évalue et sauvegarde le meilleur modèle |
| `CheckpointCallback` | Sauvegarde périodiquement |
| `StopTrainingOnRewardThreshold` | Arrête quand un reward cible est atteint |
| `CallbackList` | Combine plusieurs callbacks |
| Custom `BaseCallback` | Crée le tien |


---
## 7. Environnements vectorisés — accélérer l'entraînement

Par défaut, un seul environnement tourne à la fois. On peut en faire tourner **plusieurs en parallèle** pour collecter plus d'expériences simultanément.

In [ ]:
from stable_baselines3.common.env_util import make_vec_env

# Créer 4 environnements qui tournent en parallèle
vec_env = make_vec_env('LunarLander-v3', n_envs=4)

model = PPO('MlpPolicy', vec_env, verbose=1)
model.learn(total_timesteps=50_000)

# Attention : pour évaluer, utiliser un env non-vectorisé
eval_env = gym.make('LunarLander-v3')
mean, std = evaluate_policy(model, eval_env, n_eval_episodes=10)
print(f"Reward moyen (4 envs parallèles) : {mean:.2f} ± {std:.2f}")

vec_env.close()
eval_env.close()

### Quel impact ?

- **4 envs** → collecte 4x plus de données à chaque step
- L'entraînement est **plus stable** car les expériences sont plus diversifiées
- Idéal pour PPO et A2C (on-policy)
- Les algos off-policy (SAC, DQN) en profitent moins

---
## 8. Valider un environnement custom

In [ ]:
from stable_baselines3.common.env_checker import check_env

# À faire AVANT d'entraîner un env custom
# Vérifie que l'env respecte l'API gymnasium
env = gym.make('LunarLander-v3')
check_env(env, warn=True)
print("Environnement valide")
env.close()

---
## Récapitulatif — API SB3 en un coup d'oeil

```python
from stable_baselines3 import PPO   # ou A2C, SAC, DQN, TD3…
import gymnasium as gym

# 1. Environnement
env = gym.make('LunarLander-v3')

# 2. Modèle
model = PPO('MlpPolicy', env, verbose=1)

# 3. Entraîner
model.learn(total_timesteps=100_000)

# 4. Sauvegarder
model.save("mon_modele")

# 5. Charger
model = PPO.load("mon_modele", env=env)

# 6. Prédire
obs, _ = env.reset()
action, _ = model.predict(obs, deterministic=True)

# 7. Évaluer
from stable_baselines3.common.evaluation import evaluate_policy
mean, std = evaluate_policy(model, env, n_eval_episodes=10)
```

---

## Ce que SB3 fait automatiquement (et que tu n'as pas à coder)

| Tâche complexe | SB3 s'en occupe |
|---------------|------------------|
| Boucle d'entraînement | `model.learn()` |
| Replay buffer (off-policy) | Géré en interne |
| Normalisation des observations | Via wrappers |
| Calcul des gradients | Via PyTorch |
| Métriques d'entraînement | Loggées automatiquement |
| Compatibilité GPU | Automatique si CUDA disponible |